In [1]:
# %% PATCH 1
import pandas as pd
from sqlalchemy import create_engine
import numpy as np
import os
from dotenv import load_dotenv

# Connect to PostgreSQL
load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)


# Logging function
def log(msg):
    print(f"[INFO] {msg}")



In [2]:
# %% PATCH 2
log("Loading staging tables...")

df_hosp = pd.read_sql("SELECT * FROM staging.stg_hospitals", engine)
log(f"stg_hospitals loaded → {df_hosp.shape} rows")

df_meas = pd.read_sql("SELECT * FROM staging.stg_measures", engine)
log(f"stg_measures loaded → {df_meas.shape} rows")

df_rep = pd.read_sql("SELECT * FROM staging.stg_reported_measure_values", engine)
log(f"stg_reported_measure_values loaded → {df_rep.shape} rows")



[INFO] Loading staging tables...
[INFO] stg_hospitals loaded → (2854, 4) rows
[INFO] stg_measures loaded → (198, 4) rows
[INFO] stg_reported_measure_values loaded → (693, 6) rows


In [3]:
# %% PATCH 3
# Clean hospitals
log("Cleaning hospitals...")
before = df_hosp.shape[0]
df_hosp_clean = df_hosp.drop_duplicates(subset=["reporting_unit_code"])
after = df_hosp_clean.shape[0]
log(f"Hospitals cleaned → rows before: {before}, after: {after}")
log(f"NULLs in hospitals: \n{df_hosp_clean.isna().sum()}")

# Clean measures
log("Cleaning measures...")
before = df_meas.shape[0]
df_meas_clean = df_meas.drop_duplicates(subset=["measure_code"])
after = df_meas_clean.shape[0]
log(f"Measures cleaned → rows before: {before}, after: {after}")
log(f"NULLs in measures: \n{df_meas_clean.isna().sum()}")

# Clean reported measures
log("Cleaning reported measures...")
before = df_rep.shape[0]
df_rep_clean = df_rep.drop_duplicates(subset=["reported_measure_code", "measure_code", "data_period"])
df_rep_clean["value"] = pd.to_numeric(df_rep_clean["value"], errors="coerce")
df_rep_clean = df_rep_clean.dropna(subset=["value"])
after = df_rep_clean.shape[0]
log(f"Reported measures cleaned → rows before: {before}, after: {after}")
log(f"NULLs in reported measures: \n{df_rep_clean.isna().sum()}")



[INFO] Cleaning hospitals...
[INFO] Hospitals cleaned → rows before: 2854, after: 1427
[INFO] NULLs in hospitals: 
reporting_unit_code      0
reporting_unit_name      0
state                   21
type                   697
dtype: int64
[INFO] Cleaning measures...
[INFO] Measures cleaned → rows before: 198, after: 33
[INFO] NULLs in measures: 
measure_code    0
measure_name    0
description     0
unit            0
dtype: int64
[INFO] Cleaning reported measures...
[INFO] Reported measures cleaned → rows before: 693, after: 693
[INFO] NULLs in reported measures: 
measure_code             0
reported_measure_code    0
reported_measure_name    0
value                    0
unit                     0
data_period              0
dtype: int64


In [4]:
# %% PATCH 4
log("Simulating hospital assignment for fact table...")

df_fact_fake = df_rep_clean.merge(
    df_hosp_clean[['reporting_unit_code']],
    how='cross'
)
log(f"Fact table after simulating hospitals → {df_fact_fake.shape[0]} rows")

# Shuffle rows only
df_fact_fake = df_fact_fake.sample(frac=1, random_state=42).reset_index(drop=True)

# Shuffle values ONLY
df_fact_fake["value"] = np.random.permutation(df_fact_fake["value"].values)

log("Hospital assignment simulated. Measure–unit relationship preserved.")


[INFO] Simulating hospital assignment for fact table...
[INFO] Fact table after simulating hospitals → 988911 rows
[INFO] Hospital assignment simulated. Measure–unit relationship preserved.


In [5]:
# %% PATCH 5 - Duplicate-safe DIM table loading
log("Loading DIM tables...")

# --- DIM_HOSPITAL ---
df_hosp_clean.rename(columns={
    "reporting_unit_name": "hospital_name",
    "type": "hospital_type"
}, inplace=True)

# Check existing hospital codes
existing_hosp = pd.read_sql("SELECT reporting_unit_code FROM warehouse.dim_hospital", engine)
existing_hosp_codes = existing_hosp['reporting_unit_code'].tolist()

df_hosp_new = df_hosp_clean[~df_hosp_clean['reporting_unit_code'].isin(existing_hosp_codes)]

if not df_hosp_new.empty:
    df_hosp_new.to_sql(
        "dim_hospital",
        engine,
        schema="warehouse",
        if_exists="append",
        index=False
    )
    log(f"dim_hospital loaded → {df_hosp_new.shape[0]} new rows")
else:
    log("No new dim_hospital rows to load")


# --- DIM_MEASURE ---
df_meas_clean = df_meas_clean[[
    "measure_code",
    "measure_name",
    "description",
    "unit"
]]

existing_meas = pd.read_sql("SELECT measure_code FROM warehouse.dim_measure", engine)
existing_meas_codes = existing_meas['measure_code'].tolist()

df_meas_new = df_meas_clean[~df_meas_clean['measure_code'].isin(existing_meas_codes)]

if not df_meas_new.empty:
    df_meas_new.to_sql(
        "dim_measure",
        engine,
        schema="warehouse",
        if_exists="append",
        index=False
    )
    log(f"dim_measure loaded → {df_meas_new.shape[0]} new rows")
else:
    log("No new dim_measure rows to load")


# --- DIM_REPORTED_MEASURE ---
df_rep_dim = df_rep_clean[[
    "reported_measure_code",
    "reported_measure_name",
    "measure_code"
]]

existing_rep = pd.read_sql("SELECT reported_measure_code FROM warehouse.dim_reported_measure", engine)
existing_rep_codes = existing_rep['reported_measure_code'].tolist()

df_rep_new = df_rep_dim[~df_rep_dim['reported_measure_code'].isin(existing_rep_codes)]

if not df_rep_new.empty:
    df_rep_new.to_sql(
        "dim_reported_measure",
        engine,
        schema="warehouse",
        if_exists="append",
        index=False
    )
    log(f"dim_reported_measure loaded → {df_rep_new.shape[0]} new rows")
else:
    log("No new dim_reported_measure rows to load")


# --- DIM_TIME ---
df_time_dim = pd.DataFrame(
    df_rep_clean["data_period"].unique(),
    columns=["data_period"]
)

existing_time = pd.read_sql("SELECT data_period FROM warehouse.dim_time", engine)
existing_time_periods = existing_time['data_period'].tolist()

df_time_new = df_time_dim[~df_time_dim['data_period'].isin(existing_time_periods)]

if not df_time_new.empty:
    df_time_new.to_sql(
        "dim_time",
        engine,
        schema="warehouse",
        if_exists="append",
        index=False
    )
    log(f"dim_time loaded → {df_time_new.shape[0]} new rows")
else:
    log("No new dim_time rows to load")

[INFO] Loading DIM tables...
[INFO] No new dim_hospital rows to load
[INFO] No new dim_measure rows to load
[INFO] No new dim_reported_measure rows to load
[INFO] No new dim_time rows to load


In [ ]:
# %% PATCH 6
log("Preparing fact_measure_performance...")

# Load dimension IDs
df_hosp_ids = pd.read_sql("SELECT hospital_id, reporting_unit_code FROM warehouse.dim_hospital", engine)
df_meas_ids = pd.read_sql("SELECT measure_id, measure_code FROM warehouse.dim_measure", engine)
df_rep_ids  = pd.read_sql("SELECT reported_measure_id, reported_measure_code FROM warehouse.dim_reported_measure", engine)
df_time_ids = pd.read_sql("SELECT time_id, data_period FROM warehouse.dim_time", engine)

# Merge to get IDs
df_fact_load = (
    df_fact_fake
    .merge(df_hosp_ids, on="reporting_unit_code")
    .merge(df_meas_ids, on="measure_code")
    .merge(df_rep_ids, on="reported_measure_code")
    .merge(df_time_ids, on="data_period")
)

df_fact_load = df_fact_load[["hospital_id", "measure_id", "reported_measure_id", "time_id", "value"]]

# Load into fact table
df_fact_load.to_sql(
    "fact_measure_performance", engine, schema="warehouse", if_exists="append", index=False
)
log(f"fact_measure_performance loaded → {df_fact_load.shape[0]} rows")


[INFO] Preparing fact_measure_performance...
